In [1]:
import json
import pandas as pd
import numpy as np

train = pd.read_csv("../data/raw/train.csv")
test = pd.read_csv("../data/raw/test.csv")
print(train.shape, test.shape)

(10003, 2) (3080, 2)


In [2]:
DEPARTMENTS = {
    "Card Services": [
        "activate_my_card", "card_arrival", "card_delivery_estimate", "order_physical_card",
        "get_physical_card", "getting_spare_card", "getting_virtual_card",
        "get_disposable_virtual_card", "disposable_card_limits", "virtual_card_not_working",
        "card_not_working", "contactless_not_working", "card_about_to_expire", "card_linking",
        "card_acceptance", "card_swallowed", "visa_or_mastercard", "apple_pay_or_google_pay",
        "change_pin", "pin_blocked"],
    "Fraud & Security": [
        "compromised_card", "lost_or_stolen_card", "lost_or_stolen_phone",
        "card_payment_not_recognised", "cash_withdrawal_not_recognised",
        "direct_debit_payment_not_recognised"],
    "Payments & Cash": [
        "pending_card_payment", "declined_card_payment", "reverted_card_payment?",
        "transaction_charged_twice", "wrong_amount_of_cash_received",
        "declined_cash_withdrawal", "pending_cash_withdrawal", "atm_support"],
    "Transfers": [
        "cancel_transfer", "beneficiary_not_allowed", "pending_transfer", "failed_transfer",
        "declined_transfer", "transfer_timing", "transfer_not_received_by_recipient",
        "balance_not_updated_after_bank_transfer", "transfer_into_account", "receiving_money"],
    "Top-ups": [
        "pending_top_up", "top_up_reverted", "top_up_failed", "automatic_top_up",
        "verify_top_up", "top_up_by_cash_or_cheque", "top_up_by_card_charge",
        "top_up_by_bank_transfer_charge", "topping_up_by_card", "top_up_limits",
        "balance_not_updated_after_cheque_or_cash_deposit"],
    "Billing, Fees & Refunds": [
        "card_payment_fee_charged", "cash_withdrawal_charge", "transfer_fee_charged",
        "extra_charge_on_statement", "request_refund", "Refund_not_showing_up",
        "exchange_charge", "exchange_rate", "exchange_via_app",
        "card_payment_wrong_exchange_rate", "wrong_exchange_rate_for_cash_withdrawal"],
    "Account & Verification": [
        "why_verify_identity", "verify_my_identity", "unable_to_verify_identity",
        "verify_source_of_funds", "edit_personal_details", "terminate_account", "age_limit",
        "passcode_forgotten", "country_support", "supported_cards_and_currencies",
        "fiat_currency_support"],
}

intent_to_dept = {i: d for d, intents in DEPARTMENTS.items() for i in intents}

all_intents = set(train["category"])
print("Mapped:", len(intent_to_dept), "/", len(all_intents))
print("Missing:", all_intents - set(intent_to_dept))
print("Unknown:", set(intent_to_dept) - all_intents)

train["department"] = train["category"].map(intent_to_dept)
test["department"] = test["category"].map(intent_to_dept)

with open("../models/departments.json", "w") as f:
    json.dump(intent_to_dept, f, indent=2)

Mapped: 77 / 77
Missing: set()
Unknown: set()


In [3]:
train["department"].value_counts()

department
Card Services              2169
Billing, Fees & Refunds    1714
Top-ups                    1413
Transfers                  1409
Account & Verification     1268
Payments & Cash            1231
Fraud & Security            799
Name: count, dtype: int64

In [4]:
from sklearn.metrics import accuracy_score, f1_score

def evaluate(name, true_intents, pred_intents):
    true_d = pd.Series(true_intents).map(intent_to_dept)
    pred_d = pd.Series(pred_intents).map(intent_to_dept)
    print(f"{name:24s} | intent acc: {accuracy_score(true_intents, pred_intents)*100:5.1f}% "
          f"| intent macro-F1: {f1_score(true_intents, pred_intents, average='macro')*100:5.1f}% "
          f"| department acc: {accuracy_score(true_d, pred_d)*100:5.1f}%")

In [5]:
from sklearn.pipeline import Pipeline
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression

tfidf_lr = Pipeline([
    ("tfidf", TfidfVectorizer(ngram_range=(1, 2), min_df=2, sublinear_tf=True)),
    ("clf", LogisticRegression(max_iter=2000, C=10)),
])

tfidf_lr.fit(train["text"], train["category"])
evaluate("TF-IDF + LogReg", test["category"], tfidf_lr.predict(test["text"]))

TF-IDF + LogReg          | intent acc:  89.1% | intent macro-F1:  89.1% | department acc:  95.9%


In [6]:
pred = tfidf_lr.predict(test["text"])
wrong = test.assign(pred=pred).query("category != pred")
print("Wrong:", len(wrong), "of", len(test))
pd.Series(list(zip(wrong["category"], wrong["pred"]))).value_counts().head(10)

Wrong: 337 of 3080


(pin_blocked, get_physical_card)                                                 5
(unable_to_verify_identity, verify_my_identity)                                  5
(verify_my_identity, why_verify_identity)                                        5
(fiat_currency_support, exchange_via_app)                                        4
(pending_top_up, top_up_failed)                                                  4
(card_acceptance, card_not_working)                                              4
(top_up_reverted, top_up_failed)                                                 4
(why_verify_identity, verify_my_identity)                                        4
(wrong_exchange_rate_for_cash_withdrawal, card_payment_wrong_exchange_rate)      4
(balance_not_updated_after_bank_transfer, transfer_not_received_by_recipient)    4
Name: count, dtype: int64